<div style="background: linear-gradient(135deg, #091e3a 0%, #1e293b 50%, #0f172a 100%); padding: 28px; border-radius: 14px; border-left: 8px solid #38bdf8; color: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin-bottom: 18px; box-shadow: 0 4px 20px rgba(0,0,0,0.25);">
    <div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px; margin-bottom: 10px;">
        <span style="font-size: 12px; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: #38bdf8; background: rgba(56, 189, 248, 0.15); padding: 5px 14px; border-radius: 16px; border: 1px solid rgba(56, 189, 248, 0.3);">Kaggle Santa 2026 • Structural Audit & Baseline</span>
        <span style="font-size: 12px; color: #94a3b8; font-weight: 600;">Zero-Trust Hardware Validator • Dynamic Live Telemetry</span>
    </div>
    <h1 style="font-size: 30px; font-weight: 800; color: #ffffff; margin: 0 0 10px 0; line-height: 1.25;">Santa 2026: Structural Saturation Audit, Fast Validator & Public Union</h1>
    <p style="font-size: 14.5px; line-height: 1.6; color: #cbd5e1; margin: 0 0 14px 0; max-width: 960px;">
        An empirical architectural study of why isolated single-target cone rewrites and prefix minimizers saturate at the current public frontier—paired with a sub-millisecond 48-register validator, live multi-source public union, Shannon hardware headroom diagnostics, exact-division quotient lifting, cross-lineage Jaccard orthogonality mapping, and a drop-in submission profiler.
    </p>
</div>

In [ ]:
# Environment Setup & Quiet Imports
import os
import sys
import json
import time
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display, HTML, Markdown

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
pio.renderers.default = 'kaggle' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else 'json'
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

t_notebook_start = time.time()


## Section 1 — Executive Scorecard & Zero-Trust Public Union
A fast, self-contained **48-register hardware evaluator** enforces all Santa 2026 competition invariants (positive integers bounded by `value_limit`, exact integer division, $\le 4{,}096$ steps, $\le 128\text{ KiB}$ cell size, and $100\%$ target coverage) before auditing every mounted public dataset and forming a verified puzzle-wise union.

In [ ]:
# Standalone 48-Register Hardware Validator (Zero-Trust Specification)
class ElfCalculatorValidator:
    def __init__(self, puzzles_path: str):
        self.puzzles = {}
        df = pd.read_csv(puzzles_path)
        for _, row in df.iterrows():
            pid = str(row['id']).strip()
            if pid.isdigit():
                pid = f"scored_{int(pid):04d}"
            starts = json.loads(row['starting_values']) if isinstance(row['starting_values'], str) else list(row['starting_values'])
            targets = json.loads(row['targets']) if isinstance(row['targets'], str) else list(row['targets'])
            self.puzzles[pid] = {
                'start': starts,
                'targets': targets,
                'target_set': set(targets),
                'limit': int(row['value_limit']),
                'registers': int(row.get('registers', 48))
            }

    def validate_puzzle(self, pid: str, program: list) -> tuple[bool, str]:
        pinfo = self.puzzles.get(pid)
        if pinfo is None:
            return False, f"Unknown puzzle ID {pid}"
        if not isinstance(program, list) or len(program) > 4096:
            return False, f"Length {len(program) if isinstance(program, list) else 'N/A'} > 4096"

        regs_cap = pinfo['registers']
        limit = pinfo['limit']
        regs = [None] * regs_cap
        for idx, val in enumerate(pinfo['start']):
            regs[idx] = val

        remaining = set(pinfo['target_set']) - set(pinfo['start'])
        for step, ins in enumerate(program):
            if not isinstance(ins, (list, tuple)) or len(ins) != 4:
                return False, f"Malformed instruction at step {step}"
            op, ra, rb, rd = ins
            if not (0 <= ra < regs_cap and 0 <= rb < regs_cap and 0 <= rd < regs_cap):
                return False, f"Register out of bounds at step {step}"
            va, vb = regs[ra], regs[rb]
            if va is None or vb is None:
                return False, f"Uninitialized register at step {step}"
            if op == '+':
                out = va + vb
            elif op == '-':
                out = va - vb
                if out <= 0:
                    return False, f"Non-positive subtraction at step {step}"
            elif op == '*':
                out = va * vb
            elif op == '/':
                if vb <= 0 or va % vb != 0:
                    return False, f"Inexact division at step {step}"
                out = va // vb
            else:
                return False, f"Invalid opcode {op}"
            if out <= 0 or out > limit:
                return False, f"Value {out} out of bounds at step {step}"
            regs[rd] = out
            remaining.discard(out)

        if remaining:
            return False, f"{len(remaining)} targets unreached"
        return True, "OK"

# Locate puzzles.csv cleanly across Kaggle or local environments
puzzles_path = None
for cand in ['/kaggle/input/santa-2026/puzzles.csv', 'data/puzzles.csv', 'puzzles.csv']:
    if os.path.exists(cand):
        puzzles_path = cand
        break
if not puzzles_path and os.path.exists('/kaggle/input'):
    for root, _, files in os.walk('/kaggle/input'):
        for fn in files:
            if fn.lower() == 'puzzles.csv':
                puzzles_path = os.path.join(root, fn)
                break
        if puzzles_path:
            break

assert puzzles_path, "Could not locate puzzles.csv"
validator = ElfCalculatorValidator(puzzles_path)


In [ ]:
# Multi-Source Public Union, Zero-Trust Verification & Live KPI Banner
search_roots = ['/kaggle/input'] if (os.path.exists('/kaggle/input') and os.listdir('/kaggle/input')) else ['data/public_sweep_nov25_v21']
candidate_files = []
skip_tokens = ('puzzle', 'test', 'families', 'history', 'summary', 'sample_submission')
for sdir in search_roots:
    if os.path.exists(sdir):
        for root, _, files in os.walk(sdir):
            for fname in sorted(files):
                low = fname.lower()
                if low.endswith('.csv') and not any(tok in low for tok in skip_tokens):
                    candidate_files.append(os.path.join(root, fname))

lineage_slug_map = {
    'goodpjw2008': ['goodpjw2008', 'santa-2026-dag-lns-programs', 'santa-2026-fresh-program-re-search'],
    'ishankarnick': ['ishankarnick', 'santa-2026-frontier-programs'],
    'ozertuu': ['ozertuu', 'santa-2026-verified-programs', 'santa-2026-multi-engine-frontier'],
    'royalacecat': ['royalacecat', 'santa-2026-verified-release-artifacts', 'santa-2026-public-frontier'],
    'alexchilton': ['alexchilton', 'santa-2026-rudolph-programs', 'rudolph-the-red-nosed'],
    'asteriospanoras': ['asteriospanoras', 'santa-2026-elf-programs-and-experiments', 'santa-2026-best-score'],
    'rosswade': ['rosswade', 'santa-2026-elf-calculator-proved']
}

best_programs = {}
provenance_map = {}
all_lineage_progs = {}
best_lineage_total = {}
audit_records = []

for cpath in candidate_files:
    try:
        cdf = pd.read_csv(cpath)
        id_c = 'id' if 'id' in cdf.columns else ('puzzle_id' if 'puzzle_id' in cdf.columns else None)
        prog_c = 'program' if 'program' in cdf.columns else ('solution' if 'solution' in cdf.columns else ('instructions' if 'instructions' in cdf.columns else None))
        if not id_c or not prog_c:
            continue

        src_label = Path(cpath).parent.name + '/' + Path(cpath).name
        lin_match = None
        low_path = cpath.lower()
        for lin_name, markers in lineage_slug_map.items():
            if any(m in low_path for m in markers):
                lin_match = lin_name
                break

        c_valid = 0
        c_invalid = 0
        c_cost = 0
        lin_dict = {}

        for raw_pid, raw_prog in zip(cdf[id_c].astype(str).tolist(), cdf[prog_c].tolist()):
            pid = raw_pid.strip()
            if pid.isdigit():
                pid = f"scored_{int(pid):04d}"
            if pid not in validator.puzzles:
                continue
            try:
                ops = json.loads(raw_prog) if isinstance(raw_prog, str) else raw_prog
                if not isinstance(ops, list) or len(ops) > 4096:
                    c_invalid += 1
                    continue
                cost = len(ops)
                # Validate strictly if potentially improving or to certify lineage file
                if pid in best_programs and cost >= len(best_programs[pid]):
                    c_valid += 1
                    c_cost += cost
                    if lin_match:
                        lin_dict[pid] = ops
                    continue
                ok, _ = validator.validate_puzzle(pid, ops)
                if ok:
                    c_valid += 1
                    c_cost += cost
                    if lin_match:
                        lin_dict[pid] = ops
                    if pid not in best_programs or cost < len(best_programs[pid]):
                        best_programs[pid] = ops
                        provenance_map[pid] = src_label
                else:
                    c_invalid += 1
            except Exception:
                c_invalid += 1

        if lin_match and len(lin_dict) == len(validator.puzzles):
            if lin_match not in best_lineage_total or c_cost < best_lineage_total[lin_match]:
                best_lineage_total[lin_match] = c_cost
                all_lineage_progs[lin_match] = lin_dict

        if c_valid > 0:
            audit_records.append({
                'Source': src_label,
                'Valid': c_valid,
                'Invalid': c_invalid,
                'Score': c_cost if c_valid == len(validator.puzzles) else None
            })
    except Exception:
        continue

# Export submission.csv and perform standalone 1,000/1,000 timed verification
export_rows = [{'id': pid, 'program': json.dumps(best_programs[pid])} for pid in sorted(validator.puzzles.keys()) if pid in best_programs]
df_sub = pd.DataFrame(export_rows)
df_sub.to_csv('submission.csv', index=False)

t_val0 = time.time()
verified_total = 0
verified_count = 0
for _, r in df_sub.iterrows():
    prog = json.loads(r['program'])
    assert len(r['program'].encode('utf-8')) <= 131072, f"Cell size > 128 KiB on {r['id']}"
    ok, err = validator.validate_puzzle(r['id'], prog)
    assert ok, f"Verification failure on {r['id']}: {err}"
    verified_total += len(prog)
    verified_count += 1
val_elapsed_ms = (time.time() - t_val0) * 1000.0
val_ms_per_puzzle = val_elapsed_ms / max(1, verified_count)

banner_html = f"""
<div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); gap: 14px; margin: 12px 0 22px 0;">
    <div style="background: #0f172a; border: 1px solid #334155; border-top: 4px solid #38bdf8; border-radius: 10px; padding: 16px; color: #f8fafc;">
        <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #94a3b8;">Verified Public Score</div>
        <div style="font-size: 28px; font-weight: 800; color: #38bdf8; margin-top: 4px;">{verified_total:,} <span style="font-size: 13px; color: #cbd5e1;">ops</span></div>
        <div style="font-size: 12px; color: #94a3b8; margin-top: 4px;">Strictly public multi-source union</div>
    </div>
    <div style="background: #0f172a; border: 1px solid #334155; border-top: 4px solid #10b981; border-radius: 10px; padding: 16px; color: #f8fafc;">
        <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #94a3b8;">Zero-Trust Verification</div>
        <div style="font-size: 28px; font-weight: 800; color: #34d399; margin-top: 4px;">{verified_count:,} / 1,000</div>
        <div style="font-size: 12px; color: #94a3b8; margin-top: 4px;">0 errors • 48-reg &le;128 KiB checked</div>
    </div>
    <div style="background: #0f172a; border: 1px solid #334155; border-top: 4px solid #a855f7; border-radius: 10px; padding: 16px; color: #f8fafc;">
        <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #94a3b8;">Public Sources Audited</div>
        <div style="font-size: 28px; font-weight: 800; color: #c084fc; margin-top: 4px;">{len(audit_records)} <span style="font-size: 13px; color: #cbd5e1;">CSVs</span></div>
        <div style="font-size: 12px; color: #94a3b8; margin-top: 4px;">{len(all_lineage_progs)} complete lineages indexed</div>
    </div>
    <div style="background: #0f172a; border: 1px solid #334155; border-top: 4px solid #f59e0b; border-radius: 10px; padding: 16px; color: #f8fafc;">
        <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #94a3b8;">Validator Throughput</div>
        <div style="font-size: 28px; font-weight: 800; color: #fbbf24; margin-top: 4px;">{val_ms_per_puzzle:.2f} <span style="font-size: 13px; color: #cbd5e1;">ms/puz</span></div>
        <div style="font-size: 12px; color: #94a3b8; margin-top: 4px;">Full 1,000-puzzle audit in {val_elapsed_ms:.0f} ms</div>
    </div>
</div>
"""
display(HTML(banner_html))


## Section 2 — Sub-128.3k Structural Saturation Audit: Why Local Rewrites Stall
To understand why isolated single-target rewrites and shallow prefix searches stall below `128.3k`, we run a **bitmask ancestor-cone and SSA escape-set census** across all 1,000 verified public programs in real time:
- **Panel A (Exclusive Cone Depth vs. Shared Coupling):** Measures how many instructions belong exclusively to a single target $T$ versus shared hub nodes feeding $\ge 2$ targets.
- **Panel B (Prefix Escape-Set Tightness $|E_K|/K$):** Measures the exact live escape set ratio across setup prefix cuts $K \in \{6, 8, 10, 12, 14\}$.

In [ ]:
# Live Bitmask Exclusive-Cone & Prefix Escape-Set Saturation Audit across all 1,000 puzzles
def sim_calc(op, a, b, lim):
    if op == '+': v = a + b
    elif op == '-': v = a - b if a > b else 0
    elif op == '*': v = a * b
    elif op == '/': v = a // b if (b > 0 and a % b == 0) else 0
    else: v = 0
    return v if 0 < v <= lim else 0

cone_depths = []
shared_ratios = []
total_dag_nodes = 0
shared_dag_nodes = 0

prefix_Ks = [6, 8, 10, 12, 14]
prefix_ratios_by_K = {k: [] for k in prefix_Ks}
prefix_tight_counts = {k: 0 for k in prefix_Ks}
prefix_total_counts = {k: 0 for k in prefix_Ks}

# Also precompute 5-family categories for later sections
puzzle_meta = {}
df_puz_raw = pd.read_csv(puzzles_path)
for _, row in df_puz_raw.iterrows():
    pid = str(row['id']).strip()
    if pid.isdigit(): pid = f"scored_{int(pid):04d}"
    starts = validator.puzzles[pid]['start']
    targets = validator.puzzles[pid]['targets']
    limit = validator.puzzles[pid]['limit']
    num_t = len(targets)
    if num_t >= 30 and limit > 1e12: cat = 'independent'
    elif num_t >= 20: cat = 'mixed'
    elif limit < 1e8: cat = 'tiny'
    elif num_t <= 10: cat = 'multiplicative'
    else: cat = 'shared'

    need_cnt = len(set(targets) - set(starts))
    max_s = max(1, max(starts))
    max_t = max(1, max(targets))
    bit_gap = max(0.0, math.log2(max_t / max_s))
    shannon_floor = need_cnt + int(math.ceil(bit_gap / 3.0))
    score = len(best_programs[pid])
    puzzle_meta[pid] = {
        'category': cat,
        'num_targets': num_t,
        'limit': limit,
        'shannon_floor': shannon_floor,
        'score': score,
        'headroom': max(0, score - shannon_floor)
    }

for pid, prog in best_programs.items():
    pinfo = validator.puzzles[pid]
    starts = pinfo['start']
    targets = pinfo['targets']
    target_set = pinfo['target_set']
    limit = pinfo['limit']
    start_set = set(starts)

    # 1. Exact topological DAG + Target Bitmask Propagation
    regs = list(starts) + [0] * (48 - len(starts))
    r_src = list(range(-len(starts), 0)) + [-999] * (48 - len(starts))
    defs = {}
    topo_vals = []
    reads_at_step = []
    step_vals = []

    for s, ins in enumerate(prog):
        op, ra, rb, rd = ins
        va, vb = regs[ra], regs[rb]
        reads_at_step.append(((r_src[ra], va), (r_src[rb], vb)))
        v = sim_calc(op, va, vb, limit)
        regs[rd] = v
        r_src[rd] = s
        step_vals.append(v)
        if v not in start_set and v not in defs:
            defs[v] = (va, vb)
            topo_vals.append(v)

    # Bitmask: bit t_idx set iff node lies in target[t_idx]'s ancestor cone
    mask = defaultdict(int)
    for t_idx, t_val in enumerate(targets):
        if t_val in defs:
            mask[t_val] |= (1 << t_idx)

    for v in reversed(topo_vals):
        mv = mask.get(v, 0)
        if mv:
            va, vb = defs[v]
            if va in defs:
                mask[va] |= mv
            if vb in defs:
                mask[vb] |= mv

    active_nodes = [v for v in topo_vals if mask.get(v, 0) > 0]
    n_act = len(active_nodes)
    n_shr = sum(1 for v in active_nodes if mask[v].bit_count() >= 2)
    total_dag_nodes += n_act
    shared_dag_nodes += n_shr
    if n_act > 0:
        shared_ratios.append(n_shr / n_act)

    # Exclusive cone depth per target
    for t_idx, t_val in enumerate(targets):
        if t_val not in defs:
            cone_depths.append(0)
        else:
            t_bit = (1 << t_idx)
            c_depth = sum(1 for v in active_nodes if mask[v] == t_bit)
            cone_depths.append(c_depth)

    # 2. Exact SSA Prefix Escape-Set Tightness for K in prefix_Ks
    L = len(prog)
    for K in prefix_Ks:
        if L <= K:
            continue
        escaped = set()
        for s in range(K, L):
            (sa, va), (sb, vb) = reads_at_step[s]
            if 0 <= sa < K: escaped.add(va)
            if 0 <= sb < K: escaped.add(vb)
        later_vals = set(step_vals[K:])
        for s in range(K):
            v = step_vals[s]
            if v in target_set and v not in later_vals:
                escaped.add(v)
        escaped -= start_set
        prefix_total_counts[K] += 1
        ratio = len(escaped) / K
        prefix_ratios_by_K[K].append(ratio)
        if len(escaped) >= K:
            prefix_tight_counts[K] += 1

max_cone_depth = max(cone_depths) if cone_depths else 0
pct_cones_le4 = 100.0 * sum(1 for c in cone_depths if c <= 4) / max(1, len(cone_depths))
pct_cones_ge6 = 100.0 * sum(1 for c in cone_depths if c >= 6) / max(1, len(cone_depths))
count_cones_ge6 = sum(1 for c in cone_depths if c >= 6)
global_shared_pct = 100.0 * shared_dag_nodes / max(1, total_dag_nodes)

# Plot 2-Panel Saturation Diagnostic
fig_sat = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"<b>Panel A: Exclusive Target Cone Depth ({len(cone_depths):,} Targets)</b>",
        "<b>Panel B: Prefix Escape-Set Density |E<sub>K</sub>|/K</b>"
    )
)

cone_counts = Counter(cone_depths)
cx = sorted(cone_counts.keys())
cy = [cone_counts[x] for x in cx]
fig_sat.add_trace(
    go.Bar(x=cx, y=cy, marker_color=['#38bdf8' if x < 6 else '#f43f5e' for x in cx],
           text=[f"{y:,}" for y in cy], textposition='outside', name='Targets'),
    row=1, col=1
)

for k in prefix_Ks:
    fig_sat.add_trace(
        go.Box(y=prefix_ratios_by_K[k], name=f"K={k}", boxmean=True, marker_color='#a855f7'),
        row=1, col=2
    )
fig_sat.add_hline(y=1.0, line_dash="dash", line_color="#34d399", annotation_text="|E_K| = K Barrier", row=1, col=2)

fig_sat.update_layout(
    title="<b>Figure 1: Structural Saturation Audit — Exclusive Cones vs. Coupled Multi-Target DAGs</b>",
    template="plotly_dark",
    height=430,
    showlegend=False,
    margin=dict(l=40, r=30, t=75, b=45)
)
fig_sat.update_xaxes(title_text="Exclusive Removable Steps per Target", row=1, col=1)
fig_sat.update_yaxes(title_text="Target Count", row=1, col=1)
fig_sat.update_xaxes(title_text="Setup Prefix Length K", row=1, col=2)
fig_sat.update_yaxes(title_text="|E_K| / K Escape Ratio", row=1, col=2)
fig_sat.show()

mean_k10_tight = 100.0 * prefix_tight_counts[10] / max(1, prefix_total_counts[10])
display(HTML(
    f"<div style='background:#0f172a; border-left:4px solid #38bdf8; padding:12px 16px; border-radius:6px; color:#e2e8f0; font-size:13.5px;'>"
    f"<b>Live Saturation Takeaways:</b> Across all <b>{len(cone_depths):,}</b> evaluated targets, the maximum exclusive cone depth is just <b>{max_cone_depth} ops</b> "
    f"(<b>{pct_cones_le4:.1f}%</b> have &le;4 exclusive steps; only <b>{count_cones_ge6:,}</b> targets [{pct_cones_ge6:.1f}%] have &ge;6). "
    f"Meanwhile, <b>{global_shared_pct:.1f}%</b> of all active DAG nodes are coupled across &ge;2 targets, and <b>{mean_k10_tight:.1f}%</b> of K=10 setup prefixes already satisfy |E<sub>K</sub>| &ge; K."
    f"</div>"
))


## Section 3 — Interactive Shannon Hardware Headroom Map & 5-Family Scorecard
Plotting verified instruction counts against an information-theoretic hardware floor ($|T \setminus S| + \lceil \frac{1}{3}\log_2(\max T / \max S) \rceil$) highlights exactly where remaining solver headroom lives across the 5 puzzle families.

In [ ]:
# Interactive Shannon Headroom Scatter + Styled 5-Family Scorecard
df_meta = pd.DataFrame([{'id': pid, **d} for pid, d in puzzle_meta.items()])
fam_colors = {'independent': '#f43f5e', 'mixed': '#f59e0b', 'shared': '#38bdf8', 'multiplicative': '#10b981', 'tiny': '#a855f7'}

fig_shannon = go.Figure()
for cat in ['independent', 'mixed', 'shared', 'multiplicative', 'tiny']:
    sub = df_meta[df_meta['category'] == cat]
    if sub.empty:
        continue
    fig_shannon.add_trace(go.Scatter(
        x=sub['shannon_floor'],
        y=sub['score'],
        mode='markers',
        name=f"{cat.upper()} ({len(sub)})",
        marker=dict(size=7, color=fam_colors[cat], opacity=0.78, line=dict(width=0.5, color='#0f172a')),
        text=[f"{r['id']}<br>Score: {r['score']} ops<br>Shannon Floor: {r['shannon_floor']}<br>Headroom: {r['headroom']} ops" for _, r in sub.iterrows()],
        hoverinfo='text'
    ))

max_axis = int(max(df_meta['score'].max(), df_meta['shannon_floor'].max()) + 15)
fig_shannon.add_trace(go.Scatter(
    x=[0, max_axis], y=[0, max_axis], mode='lines',
    name='Shannon Floor (y=x)', line=dict(color='#94a3b8', dash='dash', width=1.5)
))
fig_shannon.update_layout(
    title="<b>Figure 2: Verified Instruction Count vs. Shannon Hardware Lower Bound (All 1,000 Puzzles)</b>",
    xaxis_title="Information-Theoretic Floor (ops)",
    yaxis_title="Verified Public SOTA (ops)",
    template="plotly_dark",
    height=460,
    margin=dict(l=45, r=30, t=65, b=45)
)
fig_shannon.show()

total_headroom_all = int(df_meta['headroom'].sum())
fam_rows = []
for cat, grp in df_meta.groupby('category'):
    tot_h = int(grp['headroom'].sum())
    fam_rows.append({
        'Family': cat.upper(),
        'Puzzles': len(grp),
        'Total Ops': int(grp['score'].sum()),
        'Avg Ops': grp['score'].mean(),
        'Total Headroom': tot_h,
        'Headroom Share %': 100.0 * tot_h / max(1, total_headroom_all)
    })
df_fam = pd.DataFrame(fam_rows).sort_values('Total Headroom', ascending=False).reset_index(drop=True)
styler_fam = (
    df_fam.style
    .format({'Puzzles': '{:,}', 'Total Ops': '{:,}', 'Avg Ops': '{:.1f}', 'Total Headroom': '{:,}', 'Headroom Share %': '{:.1f}%'})
    .background_gradient(subset=['Headroom Share %'], cmap='YlOrRd')
    .hide(axis='index')
)
display(HTML("<h4>5-Family Computational & Headroom Scorecard (Live Computed)</h4>"))
display(styler_fam)


## Section 4 — The Exact-Division (`/`) Anomaly & Cyclotomic Quotient Lifting
While `+`, `-`, and `*` each account for ~30%–34% of executed instructions, exact integer division `/` sits near **~2.6%**—making structured quotient lifting $T = (a \cdot b \pm c) / m$ one of the highest-leverage escapes when factorizations align.

In [ ]:
# Live Opcode Census by Family
fam_ops = defaultdict(Counter)
op_totals = Counter()
for pid, prog in best_programs.items():
    cat = puzzle_meta[pid]['category']
    for ins in prog:
        fam_ops[cat][ins[0]] += 1
        op_totals[ins[0]] += 1

total_ops_all = sum(op_totals.values())
div_pct_global = 100.0 * op_totals['/'] / max(1, total_ops_all)

cats_ord = [r['Family'].lower() for r in fam_rows]
fig_ops = go.Figure()
op_colors = {'+': '#38bdf8', '-': '#f43f5e', '*': '#a855f7', '/': '#34d399'}
for op in ['+', '-', '*', '/']:
    pcts = [100.0 * fam_ops[c][op] / max(1, sum(fam_ops[c].values())) for c in cats_ord]
    fig_ops.add_trace(go.Bar(
        name=f"Opcode [{op}]",
        x=[c.upper() for c in cats_ord],
        y=pcts,
        marker_color=op_colors[op],
        text=[f"{p:.1f}%" for p in pcts],
        textposition='auto'
    ))

fig_ops.update_layout(
    barmode='stack',
    title=f"<b>Figure 3: Opcode Distribution by Family (Global Exact-Division Share: {div_pct_global:.2f}%)</b>",
    xaxis_title="Puzzle Family",
    yaxis_title="Share of Executed Instructions (%)",
    template="plotly_dark",
    height=390,
    margin=dict(l=45, r=30, t=65, b=45)
)
fig_ops.show()


In [ ]:
# Exact-Division Quotient Lifting Demo: T = (a * b - c) / m via Cyclotomic Oracles
def verify_quotient_lift_demo():
    oracle_paths = [
        '/kaggle/input/santa-2026-math-oracles-cyclotomics/cunningham_cyclotomic_divisors.parquet',
        'data/public_sweep_nov25_v21/d_amanatar__santa-2026-math-oracles-cyclotomics/cunningham_cyclotomic_divisors.parquet'
    ]
    df_cyc = None
    for p in oracle_paths:
        if os.path.exists(p):
            df_cyc = pd.read_parquet(p)
            break

    if df_cyc is not None and not df_cyc.empty:
        sample = df_cyc.head(3)[['factor', 'base', 'n', 'sign', 'total_ops']]
        display(Markdown("**Sample Cyclotomic Divisors Mounted from `santa-2026-math-oracles-cyclotomics`:**"))
        display(sample)

    # Self-contained 48-register verification of quotient lifting:
    # Given start registers [1, 7, 13], construct target T = 5074 via T = (7^5 - 1585) / 3 ?
    # Even cleaner real identity: 7 * 13 = 91; 91 * 91 = 8281; (8281 - 1) / 8 = 1035 in just 5 ops!
    starts = [1, 7, 13]
    limit = 1000000
    lift_prog = [
        ['+', 0, 1, 3],   # r3 = 1 + 7 = 8
        ['*', 1, 2, 4],   # r4 = 7 * 13 = 91
        ['*', 4, 4, 5],   # r5 = 91 * 91 = 8281  (a * b)
        ['-', 5, 0, 6],   # r6 = 8281 - 1 = 8280 (a * b - c)
        ['/', 6, 3, 7],   # r7 = 8280 / 8 = 1035 (exact quotient lift!)
    ]
    regs = list(starts) + [0] * (48 - len(starts))
    for op, a, b, d in lift_prog:
        va, vb = regs[a], regs[b]
        regs[d] = va + vb if op == '+' else (va - vb if op == '-' else (va * vb if op == '*' else va // vb))
        assert regs[d] > 0 and regs[d] <= limit and (op != '/' or va % vb == 0)
    print(f"[*] Quotient Lift Verified: starts={starts} -> target={regs[7]} in {len(lift_prog)} ops via exact (8281 - 1) / 8!")

verify_quotient_lift_demo()


## Section 5 — Cross-Lineage Jaccard Divergence Matrix & Single-Puzzle Diff Dissector
Even when multiple public baselines tie on total score, their underlying intermediate register trajectories remain strongly orthogonal—revealing complementary paths for multi-lineage DAG recombination.

In [ ]:
# Cross-Lineage Pairwise Jaccard Divergence Heatmap & Single-Puzzle Dissector
lineages_present = [l for l in lineage_slug_map.keys() if l in all_lineage_progs]
sample_pids = sorted(validator.puzzles.keys())[:120]
lineage_val_sets = defaultdict(set)

for lin in lineages_present:
    for pid in sample_pids:
        prog = all_lineage_progs[lin].get(pid)
        if not prog:
            continue
        pinfo = validator.puzzles[pid]
        reg = list(pinfo['start']) + [0] * (48 - len(pinfo['start']))
        lim = pinfo['limit']
        for ins in prog:
            v = sim_calc(ins[0], reg[ins[1]], reg[ins[2]], lim)
            if v > 0:
                reg[ins[3]] = v
                lineage_val_sets[lin].add((pid, v))

n_lin = len(lineages_present)
div_matrix = np.zeros((n_lin, n_lin))
for i, l1 in enumerate(lineages_present):
    for j, l2 in enumerate(lineages_present):
        if i == j:
            div_matrix[i, j] = 0.0
        else:
            s1, s2 = lineage_val_sets[l1], lineage_val_sets[l2]
            div_matrix[i, j] = round(1.0 - (len(s1 & s2) / max(1, len(s1 | s2))), 3)

mean_ortho = float(np.mean(div_matrix[np.triu_indices(n_lin, k=1)])) if n_lin >= 2 else 0.0

if n_lin >= 2:
    fig_div = go.Figure(data=go.Heatmap(
        z=div_matrix,
        x=lineages_present,
        y=lineages_present,
        colorscale='Plasma',
        colorbar=dict(title='Jaccard Distance'),
        text=[[f"{div_matrix[i][j]:.1%}" for j in range(n_lin)] for i in range(n_lin)],
        texttemplate="%{text}",
        textfont={"size": 11, "color": "white"}
    ))
    fig_div.update_layout(
        title=f"<b>Figure 4: Cross-Lineage Topological Divergence Matrix (Mean Off-Diagonal: {mean_ortho:.1%})</b>",
        template="plotly_dark",
        height=470,
        margin=dict(l=60, r=40, t=65, b=55)
    )
    fig_div.show()

# Single-Puzzle Diff Dissector across all pairs of active lineages
best_diff_info = None
max_diff_j = -1.0
for i in range(n_lin):
    for j in range(i + 1, n_lin):
        la, lb = lineages_present[i], lineages_present[j]
        for pid in validator.puzzles.keys():
            pa, pb = all_lineage_progs[la].get(pid), all_lineage_progs[lb].get(pid)
            if not pa or not pb or len(pa) != len(pb) or len(pa) < 50:
                continue
            pinfo = validator.puzzles[pid]
            def trace_vals_and_regs(prog):
                reg = list(pinfo['start']) + [0] * (48 - len(pinfo['start']))
                vals, used_regs = set(), set()
                for ins in prog:
                    used_regs.add(ins[3])
                    v = sim_calc(ins[0], reg[ins[1]], reg[ins[2]], pinfo['limit'])
                    if v > 0:
                        reg[ins[3]] = v
                        vals.add(v)
                return vals, len(used_regs)
            va, ra_cnt = trace_vals_and_regs(pa)
            vb, rb_cnt = trace_vals_and_regs(pb)
            jd = 1.0 - len(va & vb) / max(1, len(va | vb))
            if jd > max_diff_j:
                max_diff_j = jd
                best_diff_info = (pid, la, lb, pa, pb, va, vb, ra_cnt, rb_cnt)

if best_diff_info:
    pid_d, la, lb, pa, pb, va, vb, ra_cnt, rb_cnt = best_diff_info
    ops_a, ops_b = Counter(ins[0] for ins in pa), Counter(ins[0] for ins in pb)
    all_ops = ['+', '-', '*', '/']
    fig_diff = go.Figure()
    fig_diff.add_trace(go.Bar(name=f"{la} (Regs Used: {ra_cnt}/48)", x=all_ops, y=[ops_a[o] for o in all_ops], marker_color='#38bdf8'))
    fig_diff.add_trace(go.Bar(name=f"{lb} (Regs Used: {rb_cnt}/48)", x=all_ops, y=[ops_b[o] for o in all_ops], marker_color='#f43f5e'))
    fig_diff.update_layout(
        barmode='group',
        title=f"<b>Figure 5: Dissected Tied-Length Puzzle {pid_d} ({len(pa)} ops • {max_diff_j:.1%} Jaccard Divergence)</b>",
        xaxis_title="Opcode",
        yaxis_title="Instruction Count",
        template="plotly_dark",
        height=370,
        margin=dict(l=45, r=30, t=65, b=45)
    )
    fig_diff.show()
    display(HTML(
        f"<div style='background:#0f172a; border-left:4px solid #f43f5e; padding:10px 15px; border-radius:6px; color:#e2e8f0; font-size:13px;'>"
        f"<b>Puzzle {pid_d} Diff:</b> Tied at <b>{len(pa)} ops</b> between <code>{la}</code> and <code>{lb}</code> with <b>{max_diff_j:.1%}</b> intermediate divergence "
        f"(Shared values: {len(va & vb)}, Unique to {la}: {len(va - vb)}, Unique to {lb}: {len(vb - va)})."
        f"</div>"
    ))


## Section 6 — Drop-In `diagnose_my_submission()` Profiler & Community Attribution
Fork this notebook and call `diagnose_my_submission('your_submission.csv')` to audit validity and compare every puzzle against the live verified public frontier.

In [ ]:
# Drop-In Submission Profiler for Community Competitors
def diagnose_my_submission(candidate_csv_path: str, top_n: int = 20):
    """Audit any candidate submission CSV against the live verified public union."""
    if not os.path.exists(candidate_csv_path):
        print(f"[!] File not found: {candidate_csv_path}")
        return
    cdf = pd.read_csv(candidate_csv_path)
    id_c = 'id' if 'id' in cdf.columns else 'puzzle_id'
    prog_c = 'program' if 'program' in cdf.columns else ('solution' if 'solution' in cdf.columns else 'instructions')

    better, matched, worse, invalid = [], [], [], []
    total_cand_ops = 0

    for _, row in cdf.iterrows():
        pid = str(row[id_c]).strip()
        if pid.isdigit():
            pid = f"scored_{int(pid):04d}"
        if pid not in validator.puzzles:
            continue
        try:
            prog = json.loads(row[prog_c]) if isinstance(row[prog_c], str) else row[prog_c]
            ok, err = validator.validate_puzzle(pid, prog)
            if not ok:
                invalid.append((pid, err))
                continue
            c_len = len(prog)
            b_len = len(best_programs.get(pid, []))
            total_cand_ops += c_len
            if c_len < b_len:
                better.append((pid, b_len, c_len, b_len - c_len))
            elif c_len == b_len:
                matched.append(pid)
            else:
                worse.append((pid, b_len, c_len, c_len - b_len))
        except Exception as e:
            invalid.append((pid, str(e)))

    print(f"\n{'='*70}\nDIAGNOSTIC REPORT: {Path(candidate_csv_path).name}\n{'='*70}")
    print(f"Total Validated Ops : {total_cand_ops:,}")
    print(f"Better than Union   : {len(better)}")
    print(f"Matched Union       : {len(matched)}")
    print(f"Worse than Union    : {len(worse)}")
    print(f"Invalid Puzzles     : {len(invalid)}")
    if better:
        print(f"\nTop {min(top_n, len(better))} Improvements:")
        for pid, bl, cl, g in sorted(better, key=lambda x: x[3], reverse=True)[:top_n]:
            print(f"  {pid}: union={bl} -> yours={cl} (-{g} ops)")
    if worse:
        print(f"\nTop {min(top_n, len(worse))} Headroom Gaps:")
        for pid, bl, cl, gap in sorted(worse, key=lambda x: x[3], reverse=True)[:top_n]:
            print(f"  {pid}: yours={cl} vs union={bl} (+{gap} ops)")
    print(f"{'='*70}\n")

print("[*] Utility `diagnose_my_submission(csv_path, top_n=20)` ready.")


In [ ]:
# Community Attribution Table & 160-DPI 2x2 Hero Dashboard Export
credit_rows = []
author_datasets = [
    ('goodpjw2008', 'goodpjw2008/santa-2026-dag-lns-programs'),
    ('ishankarnick', 'ishankarnick/santa-2026-frontier-programs'),
    ('ozertuu', 'ozertuu/santa-2026-verified-programs'),
    ('royalacecat', 'royalacecat/santa-2026-verified-release-artifacts'),
    ('alexchilton', 'alexchilton/santa-2026-rudolph-programs'),
    ('asteriospanoras', 'asteriospanoras/santa-2026-elf-programs-and-experiments'),
    ('rosswade', 'rosswade/santa-2026-elf-calculator-proved')
]
for author, slug in author_datasets:
    sc = best_lineage_total.get(author)
    ties = sum(1 for pid, p in best_programs.items() if author in all_lineage_progs and len(all_lineage_progs[author].get(pid, [])) == len(p))
    credit_rows.append({
        'Contributor': f"@{author}",
        'Public Dataset': f"https://www.kaggle.com/datasets/{slug}",
        'Verified Score': f"{sc:,}" if sc else "Mounted",
        'Ties Public Union': f"{ties:,} / 1,000" if author in all_lineage_progs else "-"
    })
display(HTML("<h4>Public Community Lineages Audited & Credited</h4>"))
display(pd.DataFrame(credit_rows).style.hide(axis='index'))

# Export 160-DPI 2x2 Static Hero Dashboard: v21_hero_dashboard.png
fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.5), dpi=160)
fig.patch.set_facecolor('#0f172a')

# 1. Shannon Headroom Map
ax1 = axes[0, 0]
ax1.set_facecolor('#1e293b')
for cat in ['independent', 'mixed', 'shared', 'multiplicative', 'tiny']:
    sub = df_meta[df_meta['category'] == cat]
    if not sub.empty:
        ax1.scatter(sub['shannon_floor'], sub['score'], s=18, alpha=0.75, color=fam_colors[cat], label=cat.upper())
ax1.plot([0, max_axis], [0, max_axis], '--', color='#94a3b8', linewidth=1.2)
ax1.set_title("1. Shannon Hardware Floor vs. Verified Score", color='#f8fafc', fontsize=11.5, fontweight='bold')
ax1.set_xlabel("Shannon Lower Bound (ops)", color='#94a3b8')
ax1.set_ylabel("Verified Score (ops)", color='#94a3b8')
ax1.tick_params(colors='#94a3b8')
ax1.legend(facecolor='#0f172a', edgecolor='#334155', labelcolor='#f8fafc', fontsize=8)

# 2. Exclusive Cone Depth & DAG Coupling
ax2 = axes[0, 1]
ax2.set_facecolor('#1e293b')
bar_cols = ['#38bdf8' if x < 6 else '#f43f5e' for x in cx]
ax2.bar(cx, cy, color=bar_cols, width=0.65)
ax2.set_title(f"2. Exclusive Target Cone Depth (Shared DAG Nodes: {global_shared_pct:.1f}%)", color='#f8fafc', fontsize=11.5, fontweight='bold')
ax2.set_xlabel("Exclusive Removable Steps per Target", color='#94a3b8')
ax2.set_ylabel("Target Count", color='#94a3b8')
ax2.tick_params(colors='#94a3b8')

# 3. Opcode / Imbalance by Family
ax3 = axes[1, 0]
ax3.set_facecolor('#1e293b')
div_pcts = [100.0 * fam_ops[c]['/'] / max(1, sum(fam_ops[c].values())) for c in cats_ord]
ax3.barh([c.upper() for c in cats_ord], div_pcts, color='#34d399', height=0.55)
ax3.axvline(div_pct_global, color='#f59e0b', linestyle='--', linewidth=1.5, label=f"Global Mean ({div_pct_global:.2f}%)")
ax3.set_title("3. Exact-Division (/) Share Across Families", color='#f8fafc', fontsize=11.5, fontweight='bold')
ax3.set_xlabel("Exact-Division Opcode Share (%)", color='#94a3b8')
ax3.tick_params(colors='#94a3b8')
ax3.legend(facecolor='#0f172a', edgecolor='#334155', labelcolor='#f8fafc', fontsize=8.5)

# 4. Cross-Lineage Jaccard Heatmap
ax4 = axes[1, 1]
ax4.set_facecolor('#1e293b')
if n_lin >= 2:
    im = ax4.imshow(div_matrix * 100.0, cmap='plasma', aspect='auto')
    ax4.set_xticks(range(n_lin))
    ax4.set_yticks(range(n_lin))
    ax4.set_xticklabels(lineages_present, rotation=30, ha='right', color='#cbd5e1', fontsize=8.5)
    ax4.set_yticklabels(lineages_present, color='#cbd5e1', fontsize=8.5)
    for i in range(n_lin):
        for j in range(n_lin):
            ax4.text(j, i, f"{div_matrix[i, j]*100:.0f}%", ha='center', va='center', color='white', fontsize=8, fontweight='bold')
ax4.set_title(f"4. Cross-Lineage Jaccard Distance (Mean: {mean_ortho:.1%})", color='#f8fafc', fontsize=11.5, fontweight='bold')

plt.tight_layout()
plt.savefig("v21_hero_dashboard.png", facecolor=fig.get_facecolor(), edgecolor='none', dpi=160)
plt.close()
print(f"[*] Exported v21_hero_dashboard.png (160 DPI) | Total runtime: {time.time()-t_notebook_start:.2f}s | Final Score: {verified_total:,}")
